In [1]:
include(dirname(dirname(pwd()))*"\\src\\TuLiPa.jl");
using .TuLiPa
using Dates, DataFrames, CSV, JSON, Plots, JuMP, HiGHS
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils.jl")
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils2.jl")

make_connection (generic function with 2 methods)

In [2]:
emps_csv_path = "./assets/emps_production_demand_nordics2.csv"
df_emps = CSV.read(emps_csv_path, DataFrame)
emps_dict = Dict{String, Dict{String, AbstractFloat}}()  # Initialize empty dataframe

# Loop through the dataframe to create a dict
for row in eachrow(df_emps)
    emps_dict[row.emps] = Dict("Power" => row.power_cost, "Power_cap" => row.power_cap, 
    "Demand" => row.demand_vol)
end

emps_dict

Dict{String, Dict{String, AbstractFloat}} with 23 entries:
  "TELEMARK"   => Dict("Power"=>50.0, "Power_cap"=>1000.0, "Demand"=>50.0)
  "SORLAND"    => Dict("Power"=>60.0, "Power_cap"=>1000.0, "Demand"=>80.0)
  "SVARTISEN"  => Dict("Power"=>30.0, "Power_cap"=>1000.0, "Demand"=>10.0)
  "SVER-SE2"   => Dict("Power"=>10.0, "Power_cap"=>1000.0, "Demand"=>40.0)
  "FINLAND"    => Dict("Power"=>100.0, "Power_cap"=>1000.0, "Demand"=>100.0)
  "MOERE"      => Dict("Power"=>40.0, "Power_cap"=>1000.0, "Demand"=>40.0)
  "OSTLAND"    => Dict("Power"=>90.0, "Power_cap"=>1000.0, "Demand"=>120.0)
  "SVER-SE1"   => Dict("Power"=>10.0, "Power_cap"=>1000.0, "Demand"=>40.0)
  "VESTMIDT"   => Dict("Power"=>10.0, "Power_cap"=>1000.0, "Demand"=>50.0)
  "HELGELAND"  => Dict("Power"=>5.0, "Power_cap"=>1000.0, "Demand"=>30.0)
  "HALLINGDAL" => Dict("Power"=>50.0, "Power_cap"=>1000.0, "Demand"=>30.0)
  "SVER-SE3"   => Dict("Power"=>80.0, "Power_cap"=>1000.0, "Demand"=>50.0)
  "DANM-DK2"   => Dict("Power"=>100.0, 

In [3]:
fix_names = Dict([
    "SE1" => "SVER-SE1", 
    "SE2" => "SVER-SE2",
    "SE3" => "SVER-SE3",
    "SE4" => "SVER-SE4",
    "DK2" => "DANM-DK2",
    "FI" => "FINLAND",
]
)

function rename_emps!(df::DataFrame, columns::Vector{Symbol}, fix_names::Dict{String, String})
    # Oppdater verdier i spesifiserte kolonner
    for col in columns
        df[!, col] .= map(x -> get(fix_names, x, x), df[!, col])
    end
    # Oppdater kolonnenavn
    rename!(df, Symbol.(map(x -> get(fix_names, String(x), String(x)), names(df))))
end

ptdf_csv_path = "./assets/ptdfs_new2.csv"
df_flowbased_grid = CSV.read(ptdf_csv_path, DataFrame)

# Bruk funksjonen på DataFrame
rename_emps!(df_flowbased_grid, [:emps_area0, :emps_area1], fix_names);



df_flowbased = process_ptdf_matrix(df_grid, false)

In [4]:
elements = create_elements(emps_dict, df_flowbased_grid);
transm = [e for e in elements if is_transmission(e)];

Dict{String, Dict{String, AbstractFloat}}("TELEMARK" => Dict("Power" => 50.0, "Power_cap" => 1000.0, "Demand" => 50.0), "SORLAND" => Dict("Power" => 60.0, "Power_cap" => 1000.0, "Demand" => 80.0), "SVARTISEN" => Dict("Power" => 30.0, "Power_cap" => 1000.0, "Demand" => 10.0), "SVER-SE2" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 40.0), "FINLAND" => Dict("Power" => 100.0, "Power_cap" => 1000.0, "Demand" => 100.0), "MOERE" => Dict("Power" => 40.0, "Power_cap" => 1000.0, "Demand" => 40.0), "OSTLAND" => Dict("Power" => 90.0, "Power_cap" => 1000.0, "Demand" => 120.0), "SVER-SE1" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 40.0), "VESTMIDT" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 50.0), "HELGELAND" => Dict("Power" => 5.0, "Power_cap" => 1000.0, "Demand" => 30.0), "HALLINGDAL" => Dict("Power" => 50.0, "Power_cap" => 1000.0, "Demand" => 30.0), "SVER-SE3" => Dict("Power" => 80.0, "Power_cap" => 1000.0, "Demand" => 50.0), "DANM-DK2" => Dict("Po

In [5]:
df_ptdf = process_ptdf_matrix(df_flowbased_grid, false);
#modelobjects = getmodelobjects(elements);

In [6]:
T = DataElement
flow_based = make_flowbased(df_ptdf, transm)
elements = vcat(elements, flow_based)



480-element Vector{DataElement}:
 DataElement("Balance", "BaseBalance", "TELEMARK", Dict{Any, Any}("Commodity" => "Power"))
 DataElement("Flow", "BaseFlow", "PowerTELEMARK", Dict{Any, Any}())
 DataElement("Arrow", "BaseArrow", "ArrowTELEMARK", Dict{Any, Any}("Balance" => "TELEMARK", "Flow" => "PowerTELEMARK", "Conversion" => 1.0, "Direction" => "In"))
 DataElement("Cost", "CostTerm", "PowerTELEMARK", Dict{Any, Any}("Param" => 50.0, "WhichInstance" => "PowerTELEMARK", "WhichConcept" => "Flow", "Direction" => "In"))
 DataElement("Capacity", "PositiveCapacity", "PowerTELEMARK_cap", Dict{Any, Any}("Param" => "PowerTELEMARK_cap", "WhichInstance" => "PowerTELEMARK", "WhichConcept" => "Flow", "Bound" => "Upper"))
 DataElement("Param", "MWToGWhSeriesParam", "PowerTELEMARK_cap", Dict{Any, Any}("Level" => 1000.0, "Profile" => 1.0))
 DataElement("Param", "MWToGWhSeriesParam", "DemandTELEMARK", Dict("Level" => 50.0, "Profile" => 1.0))
 DataElement("RHSTerm", "BaseRHSTerm", "DemandTELEMARK", Dict{A

In [7]:
#modelobjects = TuLiPa.getmodelobjects(elements)

In [8]:
elements = Vector{T}(elements)


480-element Vector{DataElement}:
 DataElement("Balance", "BaseBalance", "TELEMARK", Dict{Any, Any}("Commodity" => "Power"))
 DataElement("Flow", "BaseFlow", "PowerTELEMARK", Dict{Any, Any}())
 DataElement("Arrow", "BaseArrow", "ArrowTELEMARK", Dict{Any, Any}("Balance" => "TELEMARK", "Flow" => "PowerTELEMARK", "Conversion" => 1.0, "Direction" => "In"))
 DataElement("Cost", "CostTerm", "PowerTELEMARK", Dict{Any, Any}("Param" => 50.0, "WhichInstance" => "PowerTELEMARK", "WhichConcept" => "Flow", "Direction" => "In"))
 DataElement("Capacity", "PositiveCapacity", "PowerTELEMARK_cap", Dict{Any, Any}("Param" => "PowerTELEMARK_cap", "WhichInstance" => "PowerTELEMARK", "WhichConcept" => "Flow", "Bound" => "Upper"))
 DataElement("Param", "MWToGWhSeriesParam", "PowerTELEMARK_cap", Dict{Any, Any}("Level" => 1000.0, "Profile" => 1.0))
 DataElement("Param", "MWToGWhSeriesParam", "DemandTELEMARK", Dict("Level" => 50.0, "Profile" => 1.0))
 DataElement("RHSTerm", "BaseRHSTerm", "DemandTELEMARK", Dict{A

In [9]:
modelobjects = TuLiPa.getmodelobjects(elements)

Dict{Id, Any} with 212 entries:
  Id("FlowBased", "1331")   => BaseFlowBased{Float64}(Id("FlowBased", "1331"), …
  Id("FlowBased", "Transm_… => BaseFlowBased{Float64}(Id("FlowBased", "Transm_S…
  Id("Flow", "Transm_MOERE… => BaseFlow(Id("Flow", "Transm_MOERE->VESTMIDT"), S…
  Id("FlowBased", "Transm_… => BaseFlowBased{Float64}(Id("FlowBased", "Transm_M…
  Id("FlowBased", "2264")   => BaseFlowBased{Float64}(Id("FlowBased", "2264"), …
  Id("FlowBased", "1240")   => BaseFlowBased{Float64}(Id("FlowBased", "1240"), …
  Id("FlowBased", "2384")   => BaseFlowBased{Float64}(Id("FlowBased", "2384"), …
  Id("FlowBased", "2275")   => BaseFlowBased{Float64}(Id("FlowBased", "2275"), …
  Id("FlowBased", "Transm_… => BaseFlowBased{Float64}(Id("FlowBased", "Transm_F…
  Id("Flow", "PowerVESTMID… => BaseFlow(Id("Flow", "PowerVESTMIDT"), Sequential…
  Id("Flow", "Transm_VESTS… => BaseFlow(Id("Flow", "Transm_VESTSYD->HAUGESUND")…
  Id("Flow", "PowerTROMS")  => BaseFlow(Id("Flow", "PowerTROMS"), SequentialH

In [10]:
mymodel = JuMP.Model(HiGHS.Optimizer)
prob = TuLiPa.JuMP_Prob(modelobjects, mymodel)
datatime = getisoyearstart(2023)
scenariotime = getisoyearstart(1981)
prob_time = TuLiPa.TwoTime(datatime, scenariotime)  

update!(prob, prob_time)

In [11]:
solve!(prob)

In [1]:
#print(prob.model)

In [ ]:
-dual.(prob.model[:BalanceA]), -dual.(prob.model[:BalanceB]), -dual.(prob.model[:BalanceC])

In [2]:
#print(solution_summary(prob.model, verbose = true))

In [3]:
steps = 1
problem = prob
resultobjects = collect(values(modelobjects))
numperiods_powerhorizon = 1
numperiods_hydrohorizon = 1
periodduration_power = Day(1)
t = 1
includeexogenprice=true

prices, rhstermvalues, production, consumption, hydrolevels, batterylevels, powerbalances, rhsterms, rhstermbalances, plants, plantbalances, plantarrows, demands, demandbalances, demandarrows, hydrostorages, batterystorages  = init_results(steps, problem, modelobjects, resultobjects, numperiods_powerhorizon, numperiods_hydrohorizon, periodduration_power, t, includeexogenprice);

LoadError: UndefVarError: `prob` not defined